# Chapter 6 — Agent Skills

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

## The Hailstone Skill

The examples in this notebook use a `hailstone-sequence` skill that instructs the agent to compute a full Hailstone (Collatz) sequence using a `next_number` tool.

**Important:** Run this notebook from within the `examples/` directory so the skill is discovered as a project-scoped skill.

The skill is located at: `examples/.agents/skills/hailstone-sequence/SKILL.md`

In [ ]:
import nest_asyncio
nest_asyncio.apply()

## Examples

### Example 1: Skill Discovery

In [ ]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM

llm = OllamaLLM(model="qwen2.5:3b")
agent = LLMAgent(llm=llm)

task = Task(instruction="Compute the hailstone sequence for 6.")
handler = LLMAgent.TaskHandler(llm_agent=agent, task=task)

print(handler.skills)
print(handler.skills["hailstone-sequence"].frontmatter)
print(handler.skills["hailstone-sequence"].scope)

### Example 2: UseSkillTool Activation Result

In [ ]:
from llm_agents_from_scratch.data_structures import ToolCall
from llm_agents_from_scratch.skills.tools import UseSkillTool

tool = UseSkillTool(skills=handler.skills)
tool_call = ToolCall(
    tool_name="from_scratch__use_skill",
    arguments={"name": "hailstone-sequence"},
)
result = tool(tool_call=tool_call)
print(result.content)

### Example 3: Model-driven Skill Activation

In [ ]:
import logging
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

In [ ]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool


def next_number(x: int) -> int:
    """Performs a single step of the Hailstone sequence."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(fn=next_number)
llm = OllamaLLM(model="qwen2.5:3b")
agent = LLMAgent(llm=llm, tools=[next_number_tool])

task = Task(instruction="Compute the hailstone sequence for 6.")
result = await agent.run(task)

In [ ]:
result